# Overfitting and Underfitting From Scratch

## The Core Problem

When we train a model, we want it to learn the **real pattern** in the data —
not memorise the training examples, and not be too simple to learn anything.

Think of it like studying for an exam:

| Situation | Study habit | Result |
|-----------|-------------|--------|
| **Underfitting** | Did not study at all | Fails both practice tests and real exam |
| **Good fit** | Studied the concepts | Does well on both |
| **Overfitting** | Memorised every practice question word-for-word | Aces practice, fails real exam |

In ML terms:
- **Underfitting** — model too simple, bad on both train and test data
- **Good fit** — model learns the real pattern, good on both
- **Overfitting** — model memorises training data, great on train, bad on test

## How we detect it

We always keep some data aside (test set) that the model never sees during training.
Then we compare:

| Pattern | Train error | Test error |
|---------|------------|------------|
| Underfitting | High | High |
| Good fit | Low | Low |
| Overfitting | Very low | High |

## Step 1 — The Dataset

We generate data where marks = 5 * hours + 0.5 * hours^2 + noise.
The true relationship is a **curve** (degree 2 polynomial).

We use 40 points so the split gives us enough data to see overfitting clearly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

np.random.seed(42)

X = np.linspace(1, 8, 40)                         # 40 evenly spaced values
y = 5*X + 0.5*(X**2) + np.random.normal(0, 3, 40) # true curve + noise

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f'Training points: {len(X_train)}')
print(f'Test points:     {len(X_test)}')

plt.figure(figsize=(7, 4))
plt.scatter(X_train, y_train, label='Train', color='steelblue', zorder=5)
plt.scatter(X_test,  y_test,  label='Test',  color='tomato',    zorder=5)
plt.title('Dataset: Marks vs Hours Studied')
plt.xlabel('Hours studied')
plt.ylabel('Marks')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 2 — Underfitting (Model Too Simple)

We fit a **degree 1 polynomial** (a straight line) to data that follows a curve.
A straight line cannot capture the curve no matter how much we train it.

Result: **high error on both train and test**.

In [ ]:
def fit_poly(degree, X_train, y_train, X_test, y_test):
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train.reshape(-1, 1), y_train)
    train_mse = mean_squared_error(y_train, model.predict(X_train.reshape(-1, 1)))
    test_mse  = mean_squared_error(y_test,  model.predict(X_test.reshape(-1, 1)))
    return model, train_mse, test_mse

model_under, train_mse_under, test_mse_under = fit_poly(1, X_train, y_train, X_test, y_test)

print('UNDERFITTING MODEL (degree 1 — straight line)')
print(f'  Train MSE = {train_mse_under:.2f}')
print(f'  Test  MSE = {test_mse_under:.2f}')
print('  Both are high — model is too simple to learn the curve.')

## Step 3 — Good Fit (Right Complexity)

We fit a **degree 2 polynomial** — which matches the true shape of the data.
The model learns the curve without memorising individual points.

Result: **low error on both train and test**.

In [ ]:
model_good, train_mse_good, test_mse_good = fit_poly(2, X_train, y_train, X_test, y_test)

print('GOOD FIT MODEL (degree 2 — matches true curve)')
print(f'  Train MSE = {train_mse_good:.2f}')
print(f'  Test  MSE = {test_mse_good:.2f}')
print('  Both are low — model learned the real pattern.')

## Step 4 — Overfitting (Model Too Complex)

We fit a **degree 12 polynomial**. This model is so flexible it bends and twists
to pass through almost every training point exactly.

But those bends and twists are just fitting the **noise** — not the real pattern.
On new data (test set) it performs terribly.

Result: **very low train error, very high test error**.

In [ ]:
model_over, train_mse_over, test_mse_over = fit_poly(12, X_train, y_train, X_test, y_test)

print('OVERFITTING MODEL (degree 12 — too flexible)')
print(f'  Train MSE = {train_mse_over:.2f}   <- looks great!')
print(f'  Test  MSE = {test_mse_over:.2f}  <- terrible on new data')
print('  Model memorised the training noise, not the pattern.')

## Step 5 — Side-by-Side Comparison

Now we plot all three models on the same data to visually see the difference.
The smooth curve through the data (good fit) is what we want.
The wiggly line (overfitting) is clearly just chasing noise.

In [ ]:
x_plot = np.linspace(1, 8, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

configs = [
    (model_under, train_mse_under, test_mse_under, 'Degree 1 — Underfitting', 'orange'),
    (model_good,  train_mse_good,  test_mse_good,  'Degree 2 — Good Fit',     'green'),
    (model_over,  train_mse_over,  test_mse_over,  'Degree 12 — Overfitting', 'red'),
]

for ax, (model, tr_mse, te_mse, title, color) in zip(axes, configs):
    ax.scatter(X_train, y_train, color='steelblue', s=30, label='Train', zorder=5)
    ax.scatter(X_test,  y_test,  color='tomato',    s=30, label='Test',  zorder=5)
    ax.plot(x_plot, model.predict(x_plot), color=color, linewidth=2, label='Model')
    ax.set_title(f'{title}\nTrain MSE={tr_mse:.1f}  Test MSE={te_mse:.1f}')
    ax.set_xlabel('Hours')
    ax.set_ylabel('Marks')
    ax.legend(fontsize=8)
    ax.grid(True)
    ax.set_ylim(y.min() - 10, y.max() + 10)

plt.suptitle('Underfitting vs Good Fit vs Overfitting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 6 — The Complexity Curve (Bias-Variance Trade-off)

We train models of increasing complexity (degree 1 to 12) and track train vs test error.

This reveals the classic trade-off:
- **Left side** (simple models) — both errors are high. Underfitting.
- **Middle** — test error reaches its lowest point. Sweet spot.
- **Right side** (complex models) — train error keeps falling but test error rises. Overfitting.

The gap between train and test error is the clearest signal of overfitting.

In [ ]:
degrees = range(1, 13)
train_errors = []
test_errors  = []

for deg in degrees:
    _, tr, te = fit_poly(deg, X_train, y_train, X_test, y_test)
    train_errors.append(tr)
    test_errors.append(te)

plt.figure(figsize=(8, 5))
plt.plot(degrees, train_errors, marker='o', label='Train MSE', color='steelblue')
plt.plot(degrees, test_errors,  marker='s', label='Test MSE',  color='tomato')
plt.axvline(x=2, color='green', linestyle='--', label='Sweet spot (degree 2)')
plt.title('Train vs Test Error as Model Complexity Increases')
plt.xlabel('Polynomial Degree (model complexity)')
plt.ylabel('MSE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print('Degree | Train MSE | Test MSE')
print('-' * 35)
for deg, tr, te in zip(degrees, train_errors, test_errors):
    flag = '  <- sweet spot' if deg == 2 else ('  <- overfitting' if te > test_errors[1] * 2 else '')
    print(f'  {deg:2d}   |  {tr:7.2f}  |  {te:7.2f}{flag}')

## Step 7 — Summary Table

| | Train Error | Test Error | Problem | Fix |
|-|------------|------------|---------|-----|
| **Underfitting** | High | High | Model too simple | Use a more complex model |
| **Good fit** | Low | Low | None | Keep this |
| **Overfitting** | Very low | High | Model too complex | See below |

## How to Fix Overfitting

1. **Get more training data** — the most reliable fix. More data makes it harder to memorise.
2. **Use a simpler model** — reduce polynomial degree, fewer layers in a neural network.
3. **Regularization** — add a penalty to the loss function that punishes large weights (Ridge, Lasso).
4. **Dropout** — randomly switch off neurons during training (for neural networks).
5. **Early stopping** — stop training when test error starts going up.
6. **Cross-validation** — use multiple train/test splits to pick the right complexity.

## Key Insight

> A model that is perfect on training data is not necessarily a good model.
> Always evaluate on data the model has never seen.
> The goal is not to minimise train error — it is to minimise **test error**.